In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ludocielbeckett/health-risk-prediction-anonymized-real-data")

print("Path to dataset files:", path)

100%|██████████| 10.2k/10.2k [00:00<00:00, 18.4MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ludocielbeckett/health-risk-prediction-anonymized-real-data/versions/1


In [1]:
print("hello")

hello


In [3]:
import os
os.listdir(path)

['Health_Risk_Dataset.csv']

In [4]:
data_path = os.path.join(path,"Health_Risk_Dataset.csv")

In [5]:
import pandas as pd

df = pd.read_csv(data_path)
df

,Patient_ID,Respiratory_Rate,Oxygen_Saturation,O2_Scale,Systolic_BP,Heart_Rate,Temperature,Consciousness,On_Oxygen,Risk_Level
0,P0522,25,96,1,97,107,37.5,A,0,Medium
1,P0738,28,92,2,116,151,38.5,P,1,High
2,P0741,29,91,1,79,135,38.4,A,0,High
3,P0661,24,96,1,95,92,37.3,A,0,Medium
4,P0412,20,96,1,97,97,37.4,A,0,Low
...,...,...,...,...,...,...,...,...,...,...
995,P0107,18,96,1,124,72,37.2,A,0,Normal
996,P0271,19,95,1,129,70,38.1,A,0,Low
997,P0861,39,82,1,101,137,37.5,A,1,High
998,P0436,15,92,1,106,108,38.6,A,0,Medium


In [6]:
from xgboost import XGBClassifier

df.columns

Index(['Patient_ID', 'Respiratory_Rate', 'Oxygen_Saturation', 'O2_Scale',
       'Systolic_BP', 'Heart_Rate', 'Temperature', 'Consciousness',
       'On_Oxygen', 'Risk_Level'],
      dtype='object')

In [8]:
import numpy as np
df['Consciousness'] = np.where(df['Consciousness'] == "A", 1, 0)

y = np.where(
    df['Risk_Level'] == "High", 2,
    np.where(df['Risk_Level'] == "Medium", 1, 0)
)

X = df[['Respiratory_Rate', 'Oxygen_Saturation', 'O2_Scale',
       'Systolic_BP', 'Heart_Rate', 'Temperature', 'Consciousness',
       'On_Oxygen']]

In [12]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y)

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=3,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1, num_class=4, ...)

In [14]:
y_pred = model.predict(X_test)

score = accuracy_score(y_test,y_pred)
score

0.985

In [18]:
import joblib

joblib.dump(model,"xgboost_heart_disease.joblib")

['xgboost_heart_disease.joblib']

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("miadul/lifestyle-and-health-risk-prediction")

print("Path to dataset files:", path)

100%|██████████| 59.7k/59.7k [00:00<00:00, 11.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/miadul/lifestyle-and-health-risk-prediction/versions/1


In [17]:
os.listdir(path)

['Lifestyle_and_Health_Risk_Prediction_Synthetic_Dataset.csv']

In [20]:
data_path = os.path.join(path,'Lifestyle_and_Health_Risk_Prediction_Synthetic_Dataset.csv')
df = pd.read_csv(data_path)
df

,age,weight,height,exercise,sleep,sugar_intake,smoking,alcohol,married,profession,bmi,health_risk
0,56,67,195,low,6.1,medium,yes,yes,yes,office_worker,17.6,high
1,69,76,170,high,6.9,high,no,no,no,teacher,26.3,high
2,46,106,153,high,6.6,low,yes,no,no,artist,45.3,high
3,32,54,186,medium,8.5,medium,no,no,no,artist,15.6,low
4,60,98,195,high,8.0,low,no,no,yes,teacher,25.8,high
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,42,75,187,medium,7.2,low,no,no,yes,driver,21.4,low
4996,39,86,183,low,7.4,low,no,no,yes,engineer,25.7,high
4997,48,56,194,none,7.2,high,yes,yes,yes,engineer,14.9,high
4998,34,109,148,medium,9.9,medium,no,no,yes,doctor,49.8,low


In [21]:
df['profession'].unique()

array(['office_worker', 'teacher', 'artist', 'farmer', 'driver',
       'engineer', 'student', 'doctor'], dtype=object)

In [28]:
import numpy as np
import pandas as pd

df['exercise'] = np.where(df['exercise'] == 'low', 0, np.where(df['exercise'] == 'medium', 1, 2))
df['sugar_intake'] = np.where(df['sugar_intake'] == 'low', 0, np.where(df['sugar_intake'] == 'medium', 1, 2))
df['smoking'] = np.where(df['smoking'] == 'no', 0, 1)
df['alcohol'] = np.where(df['alcohol'] == 'no', 0, 1)
df['married'] = np.where(df['married'] == 'no', 0, 1)
df['profession'] = np.where(df['profession'] == 'office_worker', 0, np.where(df['profession'] == 'teacher', 1, np.where(df['profession'] == 'artist', 2, np.where(df['profession'] == 'farmer', 3, np.where(df['profession'] == 'driver', 4, np.where(df['profession'] == 'engineer', 5, np.where(df['profession'] == 'student', 6, 7)))))))
y = np.where(df['health_risk'] == 'low', 0, 1)

X = df[['age', 'weight', 'height', 'exercise', 'sleep', 'sugar_intake', 'smoking', 'alcohol', 'married', 'profession', 'bmi']]

print(X.head())
print(y[:5])

   age  weight  height  exercise  sleep  sugar_intake  smoking  alcohol  \
0   56      67     195         2    6.1             2        1        1   
1   69      76     170         2    6.9             2        1        1   
2   46     106     153         2    6.6             2        1        1   
3   32      54     186         2    8.5             2        1        1   
4   60      98     195         2    8.0             2        1        1   

   married  profession   bmi  
0        1           7  17.6  
1        1           7  26.3  
2        1           7  45.3  
3        1           7  15.6  
4        1           7  25.8  
[1 1 1 0 1]


In [34]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

scaler = StandardScaler()

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=3,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

score = accuracy_score(y_test, y_pred)

print("Accuracy:", score)
print(y_pred[:20])

Accuracy: 0.827
[1 1 1 0 1 0 0 1 1 1 1 0 1 1 1 1 1 0 1 1]


In [35]:
model.fit(X_train_scaled,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0.1,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

In [37]:
y_pred = model.predict(X_test_scaled)
y_pred
score = accuracy_score(y_test,y_pred)
score

0.827

In [38]:
joblib.dump(model,'xgboost_risk.joblib')

['xgboost_risk.joblib']

In [1]:
# covid dataset

import kagglehub

# Download latest version
path = kagglehub.dataset_download("sachinkumar413/covid-pneumonia-normal-chest-xray-images")

print("Path to dataset files:", path)

100%|██████████| 277M/277M [00:02<00:00, 138MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sachinkumar413/covid-pneumonia-normal-chest-xray-images/versions/1


In [4]:
import os
os.listdir(path)
covid_path = os.path.join(path,'COVID')
normal_path = os.path.join(path,'NORMAL')
pneumonia_path = os.path.join(path,'PNEUMONIA')
os.listdir(covid_path)

['COVID_1596.png',
 'COVID_1063.png',
 'COVID_233.png',
 'COVID_822.png',
 'COVID_658.png',
 'COVID_29.png',
 'COVID_86.png',
 'COVID_1085.png',
 'COVID_775.png',
 'COVID_30.png',
 'COVID_940.png',
 'COVID_1295.png',
 'COVID_1107.png',
 'COVID_769.png',
 'COVID_592.png',
 'COVID_89.png',
 'COVID_871.png',
 'COVID_133.png',
 'COVID_131.png',
 'COVID_1566.png',
 'COVID_1527.png',
 'COVID_1016.png',
 'COVID_1511.png',
 'COVID_107.png',
 'COVID_639.png',
 'COVID_727.png',
 'COVID_774.png',
 'COVID_1311.png',
 'COVID_333.png',
 'COVID_1516.png',
 'COVID_1162.png',
 'COVID_14.png',
 'COVID_1201.png',
 'COVID_1125.png',
 'COVID_171.png',
 'COVID_702.png',
 'COVID_1360.png',
 'COVID_961.png',
 'COVID_810.png',
 'COVID_1041.png',
 'COVID_1393.png',
 'COVID_345.png',
 'COVID_164.png',
 'COVID_1455.png',
 'COVID_331.png',
 'COVID_1194.png',
 'COVID_1280.png',
 'COVID_1537.png',
 'COVID_504.png',
 'COVID_441.png',
 'COVID_101.png',
 'COVID_1276.png',
 'COVID_956.png',
 'COVID_525.png',
 'COVID_10.

In [5]:
import os
import cv2
import numpy as np

X = []
y = []

classes = {'NORMAL': 0, 'PNEUMONIA': 1, 'COVID': 2}

for folder, label in classes.items():
    folder_path = os.path.join(path, folder)
    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (224, 224))
            X.append(img)
            y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y)

X = X.reshape(-1, 224, 224, 1)

print(X.shape)
print(y.shape)

(5228, 224, 224, 1)
(5228,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
history = model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=7,batch_size=32)
loss,accuracy = model.evaluate(X_test,y_test)

Epoch 1/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.9756 - loss: 0.0665 - val_accuracy: 0.9522 - val_loss: 0.1873
Epoch 2/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.9797 - loss: 0.0640 - val_accuracy: 0.9598 - val_loss: 0.1368
Epoch 3/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.9873 - loss: 0.0395 - val_accuracy: 0.9598 - val_loss: 0.1405
Epoch 4/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 54ms/step - accuracy: 0.9840 - loss: 0.0422 - val_accuracy: 0.9665 - val_loss: 0.1424
Epoch 5/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.9847 - loss: 0.0415 - val_accuracy: 0.9618 - val_loss: 0.1434
Epoch 6/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.9888 - loss: 0.0345 - val_accuracy: 0.9589 - val_loss: 0.1686
Epoch 7/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - accuracy: 0.9933 - loss: 0.0186 - val_accuracy: 0.9570 - val_loss: 0.2709
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9570 - loss: 0.2709


In [11]:
y_pred = np.argmax(model.predict(X_test), axis=1)
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred, target_names=['NORMAL', 'PNEUMONIA', 'COVID']))
print(confusion_matrix(y_test, y_pred))

33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step
              precision    recall  f1-score   support

      NORMAL       0.90      0.99      0.94       361
   PNEUMONIA       0.98      0.91      0.95       360
       COVID       1.00      0.98      0.99       325

    accuracy                           0.96      1046
   macro avg       0.96      0.96      0.96      1046
weighted avg       0.96      0.96      0.96      1046

[[356   5   0]
 [ 31 328   1]
 [  7   1 317]]


In [14]:
s = model.save("chest_xray_model.h5")

In [2]:
# brain tumor model
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


In [3]:
os.listdir(path)

['Training', 'Testing']

In [4]:
import tensorflow as tf
import os

train_path = os.path.join(path,'Training')
test_path = os.path.join(path,'Testing')

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(224,224),
    batch_size=32,
    validation_split=0.2,
    subset='training',
    seed=42,
    label_mode='int'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(224,224),
    batch_size=32,
    validation_split=0.2,
    subset='validation',
    seed=42,
    label_mode='int'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224,224),
    batch_size=32,
    shuffle=False,
    label_mode='int'
)

print(train_ds.class_names)

normalization = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y:(normalization(x),y))
val_ds = val_ds.map(lambda x,y:(normalization(x),y))
test_ds = test_ds.map(lambda x,y:(normalization(x),y))

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Dropout

model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    MaxPooling2D((2,2)),
    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D((2,2)),
    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),
    Dense(4,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

loss,accuracy = model.evaluate(test_ds)

print("Test Accuracy:",accuracy)

model.save("brain_tumor_model.h5")

from sklearn.metrics import classification_report,confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images,labels in test_ds:
    predictions = model.predict(images,verbose=0)
    predictions = np.argmax(predictions,axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(predictions)

print(classification_report(
    y_true,
    y_pred,
    target_names=['glioma','meningioma','notumor','pituitary']
))

print(confusion_matrix(y_true,y_pred))

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.
['glioma', 'meningioma', 'notumor', 'pituitary']


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 25s 120ms/step - accuracy: 0.6478 - loss: 0.8651 - val_accuracy: 0.8348 - val_loss: 0.4517
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 71ms/step - accuracy: 0.8192 - loss: 0.4564 - val_accuracy: 0.8866 - val_loss: 0.3381
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step - accuracy: 0.8766 - loss: 0.3340 - val_accuracy: 0.9054 - val_loss: 0.3002
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step - accuracy: 0.9027 - loss: 0.2596 - val_accuracy: 0.8946 - val_loss: 0.2928
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 9s 64ms/step - accuracy: 0.9250 - loss: 0.1990 - val_accuracy: 0.9179 - val_loss: 0.2231
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step - accuracy: 0.9547 - loss: 0.1345 - val_accuracy: 0.9348 - val_loss: 0.2200
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 70ms/step - accuracy: 0.9609 - loss: 0.1058 - val_accuracy: 0.9384 - val_loss: 0.2033
Epoch 8/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 10s 72ms/step - accuracy: 0.9641 - loss: 0.0949 - 

Test Accuracy: 0.8725000023841858
              precision    recall  f1-score   support

      glioma       0.85      0.77      0.81       400
  meningioma       0.86      0.81      0.83       400
     notumor       0.84      0.99      0.91       400
   pituitary       0.94      0.92      0.93       400

    accuracy                           0.87      1600
   macro avg       0.87      0.87      0.87      1600
weighted avg       0.87      0.87      0.87      1600

[[307  36  44  13]
 [ 34 323  33  10]
 [  1   1 398   0]
 [ 18  14   0 368]]


In [5]:
model.save("brain_mri_model.h5")

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Path to dataset files: /kaggle/input/skin-cancer-mnist-ham10000


In [7]:
os.listdir(path)

['hmnist_8_8_RGB.csv',
 'hmnist_28_28_RGB.csv',
 'HAM10000_images_part_1',
 'ham10000_images_part_1',
 'hmnist_8_8_L.csv',
 'HAM10000_images_part_2',
 'ham10000_images_part_2',
 'hmnist_28_28_L.csv',
 'HAM10000_metadata.csv']

In [9]:
import kagglehub
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

metadata_path = os.path.join(path,"HAM10000_metadata.csv")
folder1 = os.path.join(path,"HAM10000_images_part_1")
folder2 = os.path.join(path,"HAM10000_images_part_2")

df = pd.read_csv(metadata_path)

def find_image(image_id):
    name = image_id + ".jpg"
    p1 = os.path.join(folder1,name)
    p2 = os.path.join(folder2,name)
    return p1 if os.path.exists(p1) else p2 if os.path.exists(p2) else None

df["image_path"] = df["image_id"].apply(find_image)
df = df.dropna(subset=["image_path"])

classes = {"akiec":0,"bcc":1,"bkl":2,"df":3,"mel":4,"nv":5,"vasc":6}
df["label"] = df["dx"].map(classes)

train_df,temp_df = train_test_split(df,test_size=0.2,stratify=df["label"],random_state=42)
val_df,test_df = train_test_split(temp_df,test_size=0.5,stratify=temp_df["label"],random_state=42)

def load_image(path,label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image,channels=3)
    image = tf.image.resize(image,(224,224))
    image = image / 255.0
    return image,label

def create_dataset(data,shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((data["image_path"].values,data["label"].values))
    ds = ds.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000)
    return ds.batch(16).prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_df,True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

model = tf.keras.Sequential([
    augmentation,
    tf.keras.layers.Conv2D(32,(3,3),activation="relu",input_shape=(224,224,3)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(128,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(256,(3,3),activation="relu"),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256,activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(7,activation="softmax")
])

model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

history = model.fit(train_ds,validation_data=val_ds,epochs=7)

loss,accuracy = model.evaluate(test_ds)
print("Test Accuracy:",accuracy)

y_true = []
y_pred = []

for images,labels in test_ds:
    predictions = np.argmax(model.predict(images,verbose=0),axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(predictions)

class_names = ["akiec","bcc","bkl","df","mel","nv","vasc"]

print(classification_report(y_true,y_pred,target_names=class_names))
print(confusion_matrix(y_true,y_pred))

model.save("skin_lesion_model.h5")
print("Model saved successfully!")

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Epoch 1/7


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


501/501 ━━━━━━━━━━━━━━━━━━━━ 58s 102ms/step - accuracy: 0.6690 - loss: 1.0296 - val_accuracy: 0.6693 - val_loss: 0.9598
Epoch 2/7
501/501 ━━━━━━━━━━━━━━━━━━━━ 52s 92ms/step - accuracy: 0.6719 - loss: 0.9697 - val_accuracy: 0.6713 - val_loss: 0.9396
Epoch 3/7
501/501 ━━━━━━━━━━━━━━━━━━━━ 49s 87ms/step - accuracy: 0.6699 - loss: 0.9320 - val_accuracy: 0.6813 - val_loss: 0.9248
Epoch 4/7
501/501 ━━━━━━━━━━━━━━━━━━━━ 57s 106ms/step - accuracy: 0.6796 - loss: 0.9020 - val_accuracy: 0.6853 - val_loss: 0.8506
Epoch 5/7
501/501 ━━━━━━━━━━━━━━━━━━━━ 53s 95ms/step - accuracy: 0.6842 - loss: 0.8583 - val_accuracy: 0.6883 - val_loss: 0.8576
Epoch 6/7
501/501 ━━━━━━━━━━━━━━━━━━━━ 50s 90ms/step - accuracy: 0.6921 - loss: 0.8368 - val_accuracy: 0.6983 - val_loss: 0.8124
Epoch 7/7
501/501 ━━━━━━━━━━━━━━━━━━━━ 50s 89ms/step - accuracy: 0.6998 - loss: 0.8178 - val_accuracy: 0.7173 - val_loss: 0.7951
63/63 ━━━━━━━━━━━━━━━━━━━━ 8s 126ms/step - accuracy: 0.7116 - loss: 0.7893
Test Accuracy: 0.7115768194198

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

       akiec       0.00      0.00      0.00        32
         bcc       0.34      0.21      0.26        52
         bkl       0.29      0.41      0.34       110
          df       0.00      0.00      0.00        11
         mel       0.46      0.16      0.24       112
          nv       0.83      0.94      0.88       671
        vasc       0.53      0.64      0.58        14

    accuracy                           0.71      1002
   macro avg       0.35      0.34      0.33      1002
weighted avg       0.66      0.71      0.68      1002

[[  0   3  24   0   1   4   0]
 [  0  11  25   0   1  12   3]
 [  0  10  45   0   8  47   0]
 [  0   1   5   0   0   4   1]
 [  0   1  32   0  18  61   0]
 [  0   5  21   0  11 630   4]
 [  0   1   2   0   0   2   9]]
Model saved successfully!


In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dankok/diabetic-retinopathy-image-dataset")

print("Path to dataset files:", path)

100%|██████████| 408M/408M [00:06<00:00, 68.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dankok/diabetic-retinopathy-image-dataset/versions/1


In [11]:
os.listdir(path)

['Localization', 'Disease_Grading', 'Segmentation']

In [20]:
import kagglehub
import os

path = kagglehub.dataset_download("dankok/diabetic-retinopathy-image-dataset")

print("Dataset path:",path)

localization_path = os.path.join(path,"Localization","Localization")
disease_grading_path = os.path.join(path,"Disease_Grading")
segmentation_path = os.path.join(path,"Segmentation")

localization_groundtruth_path = os.path.join(localization_path,"Groundtruths")
localization_images_path = os.path.join(localization_path,"Original_Images")

localization_train_path = os.path.join(localization_images_path,"Training Set")
localization_test_path = os.path.join(localization_images_path,"Testing Set")

disease_groundtruth_path = os.path.join(disease_grading_path,"Groundtruths")
disease_images_path = os.path.join(disease_grading_path,"Original_Images")

disease_train_path = os.path.join(disease_images_path,"Training Set")
disease_test_path = os.path.join(disease_images_path,"Testing Set")

segmentation_groundtruth_path = os.path.join(segmentation_path,"Segmentation_Groundtruths")
segmentation_images_path = os.path.join(segmentation_path,"Original_Images")

print("Localization:",os.listdir(localization_path))
print("Localization Training:",os.listdir(localization_train_path))
print("Localization Testing:",os.listdir(localization_test_path))

print("Disease Grading:",os.listdir(disease_grading_path))
print("Disease Training:",os.listdir(disease_train_path))
print("Disease Testing:",os.listdir(disease_test_path))

print("Segmentation:",os.listdir(segmentation_path))
print("Segmentation Images:",os.listdir(segmentation_images_path))
print("Segmentation Groundtruths:",os.listdir(segmentation_groundtruth_path))

Using Colab cache for faster access to the 'diabetic-retinopathy-image-dataset' dataset.
Dataset path: /kaggle/input/diabetic-retinopathy-image-dataset
Localization: ['Original_Images', 'Groundtruths']
Localization Training: ['IDRiD_179.jpg', 'IDRiD_193.jpg', 'IDRiD_316.jpg', 'IDRiD_407.jpg', 'IDRiD_335.jpg', 'IDRiD_329.jpg', 'IDRiD_075.jpg', 'IDRiD_054.jpg', 'IDRiD_239.jpg', 'IDRiD_243.jpg', 'IDRiD_042.jpg', 'IDRiD_297.jpg', 'IDRiD_383.jpg', 'IDRiD_299.jpg', 'IDRiD_181.jpg', 'IDRiD_323.jpg', 'IDRiD_242.jpg', 'IDRiD_251.jpg', 'IDRiD_401.jpg', 'IDRiD_390.jpg', 'IDRiD_378.jpg', 'IDRiD_170.jpg', 'IDRiD_398.jpg', 'IDRiD_059.jpg', 'IDRiD_280.jpg', 'IDRiD_073.jpg', 'IDRiD_049.jpg', 'IDRiD_212.jpg', 'IDRiD_094.jpg', 'IDRiD_121.jpg', 'IDRiD_012.jpg', 'IDRiD_368.jpg', 'IDRiD_367.jpg', 'IDRiD_163.jpg', 'IDRiD_099.jpg', 'IDRiD_314.jpg', 'IDRiD_027.jpg', 'IDRiD_384.jpg', 'IDRiD_274.jpg', 'IDRiD_339.jpg', 'IDRiD_110.jpg', 'IDRiD_409.jpg', 'IDRiD_322.jpg', 'IDRiD_009.jpg', 'IDRiD_319.jpg', 'IDRiD_28

In [27]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU available:", gpus)
    try:
        for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
    except: pass
else:
    print("WARNING: GPU not detected.")

path = kagglehub.dataset_download("dankok/diabetic-retinopathy-image-dataset")
print("Dataset path:", path)

disease_grading_path = os.path.join(path, "Disease_Grading")
groundtruth_path = os.path.join(disease_grading_path, "Groundtruths")
images_path = os.path.join(disease_grading_path, "Original_Images")
train_images_path = os.path.join(images_path, "Training Set")
test_images_path = os.path.join(images_path, "Testing Set")

train_csv = os.path.join(groundtruth_path, "IDRiD_Disease Grading_Training Labels.csv")
test_csv = os.path.join(groundtruth_path, "IDRiD_Disease Grading_Testing Labels.csv")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

train_df = train_df[["Image name", "Retinopathy grade", "Risk of macular edema"]].copy()
test_df = test_df[["Image name", "Retinopathy grade", "Risk of macular edema"]].copy()

def get_train_path(image_name):
    image_name = str(image_name).strip()
    if not image_name.lower().endswith(".jpg"): image_name += ".jpg"
    return os.path.join(train_images_path, image_name)

def get_test_path(image_name):
    image_name = str(image_name).strip()
    if not image_name.lower().endswith(".jpg"): image_name += ".jpg"
    return os.path.join(test_images_path, image_name)

train_df["image_path"] = train_df["Image name"].apply(get_train_path)
test_df["image_path"] = test_df["Image name"].apply(get_test_path)

train_df["exists"] = train_df["image_path"].apply(os.path.exists)
test_df["exists"] = test_df["image_path"].apply(os.path.exists)

print("Training images found:", train_df["exists"].sum())
print("Training images missing:", (~train_df["exists"]).sum())
print("Testing images found:", test_df["exists"].sum())
print("Testing images missing:", (~test_df["exists"]).sum())

train_df = train_df[train_df["exists"]].copy()
test_df = test_df[test_df["exists"]].copy()

train_df["label"] = train_df["Retinopathy grade"].astype(int)
test_df["label"] = test_df["Retinopathy grade"].astype(int)

train_df, val_df = train_test_split(train_df, test_size=0.20, stratify=train_df["label"], random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print(train_df["label"].value_counts().sort_index())

classes = np.unique(train_df["label"])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"])
class_weights = {int(cls): float(weight) for cls, weight in zip(classes, weights)}

print("Class weights:", class_weights)

IMG_SIZE = 256
BATCH_SIZE = 16
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
    image = tf.cast(image, tf.float32)
    return image, label

def create_dataset(dataframe, shuffle=False):
    paths = dataframe["image_path"].values
    labels = dataframe["label"].values.astype(np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle: dataset = dataset.shuffle(buffer_size=len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset

train_ds = create_dataset(train_df, shuffle=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(height_factor=(-0.10, 0.10), width_factor=(-0.10, 0.10)),
    tf.keras.layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    tf.keras.layers.RandomContrast(0.15)
])

from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="retinal_image")
x = augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.45)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="dr_grade")(x)

model = tf.keras.Model(inputs, outputs, name="IDRiD_EfficientNetB0")

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])

model.summary()

checkpoint_path = "/content/best_retinopathy_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1)
]

history_stage1 = model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weights, callbacks=callbacks)

base_model.trainable = True

fine_tune_from = 180

for layer in base_model.layers[:fine_tune_from]: layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization): layer.trainable = False

print("Trainable layers:", sum(layer.trainable for layer in base_model.layers))

LABEL_SMOOTHING = 0.05

def sparse_label_smoothing_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
    y_true = y_true * (1.0 - LABEL_SMOOTHING) + LABEL_SMOOTHING / NUM_CLASSES
    y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
    loss = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
    return tf.reduce_mean(loss)

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss=sparse_label_smoothing_loss, metrics=["accuracy"])

history_stage2 = model.fit(train_ds, validation_data=val_ds, epochs=5, class_weight=class_weights, callbacks=callbacks)

model = tf.keras.models.load_model(checkpoint_path, custom_objects={"sparse_label_smoothing_loss": sparse_label_smoothing_loss})

test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Accuracy:", test_accuracy * 100, "%")

y_true = []
y_pred = []
probabilities = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    predicted_classes = np.argmax(preds, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)
    probabilities.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
probabilities = np.array(probabilities)

class_names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

print(classification_report(y_true, y_pred, labels=[0, 1, 2, 3, 4], target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])

print("Confusion Matrix:")
print(cm)

balanced_acc = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("Balanced Accuracy:", balanced_acc)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

for i, name in enumerate(class_names):
    mask = y_true == i
    if mask.sum() > 0:
        accuracy = (y_pred[mask] == i).mean()
        print(f"{i} - {name}: {accuracy * 100:.2f}% ({mask.sum()} samples)")

final_model_path = "/content/IDRiD_EfficientNetB0_DR_Classifier.keras"

model.save(final_model_path)

np.save("/content/class_names.npy", np.array(class_names))

print("Model saved successfully:", final_model_path)
print("Class names saved successfully.")

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Using Colab cache for faster access to the 'diabetic-retinopathy-image-dataset' dataset.
Dataset path: /kaggle/input/diabetic-retinopathy-image-dataset
Training images found: 413
Training images missing: 0
Testing images found: 103
Testing images missing: 0
Train: 330
Validation: 83
Test: 103
label
0    107
1     16
2    109
3     59
4     39
Name: count, dtype: int64
Class weights: {0: 0.616822429906542, 1: 4.125, 2: 0.6055045871559633, 3: 1.11864406779661, 4: 1.6923076923076923}


Model: "IDRiD_EfficientNetB0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ retinal_image (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_11 (Sequential)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 8, 8, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dr_grade (Dense)                │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,383,912 (16.72 MB)

 Trainable params: 331,781 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/5
20/21 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.2343 - loss: 2.2332
Epoch 1: val_loss improved from None to 1.32714, saving model to /content/best_retinopathy_model.keras

Epoch 1: finished saving model to /content/best_retinopathy_model.keras
21/21 ━━━━━━━━━━━━━━━━━━━━ 34s 485ms/step - accuracy: 0.2606 - loss: 2.0760 - val_accuracy: 0.4699 - val_loss: 1.3271 - learning_rate: 0.0010
Epoch 2/5
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4664 - loss: 1.7592
Epoch 2: val_loss did not improve from 1.32714
21/21 ━━━━━━━━━━━━━━━━━━━━ 20s 326ms/step - accuracy: 0.4515 - loss: 1.7753 - val_accuracy: 0.3253 - val_loss: 1.3287 - learning_rate: 0.0010
Epoch 3/5
20/21 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.4980 - loss: 1.3043
Epoch 3: val_loss did not improve from 1.32714
21/21 ━━━━━━━━━━━━━━━━━━━━ 20s 316ms/step - accuracy: 0.4636 - loss: 1.4518 - val_accuracy: 0.3494 - val_loss: 1.3895 - learning_rate: 0.0010
Epoch 4/5
20/21 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - 

                  precision    recall  f1-score   support

           No DR       0.62      0.59      0.61        34
            Mild       0.00      0.00      0.00         5
        Moderate       0.39      0.59      0.47        32
          Severe       0.40      0.21      0.28        19
Proliferative DR       0.50      0.46      0.48        13

        accuracy                           0.48       103
       macro avg       0.38      0.37      0.37       103
    weighted avg       0.46      0.48      0.46       103

Confusion Matrix:
[[20  0 14  0  0]
 [ 2  0  3  0  0]
 [ 5  0 19  6  2]
 [ 3  0  8  4  4]
 [ 2  0  5  0  6]]
Balanced Accuracy: 0.3708100142891165
Macro F1: 0.3662116954990518
Weighted F1: 0.4572794718001726
0 - No DR: 58.82% (34 samples)
1 - Mild: 0.00% (5 samples)
2 - Moderate: 59.38% (32 samples)
3 - Severe: 21.05% (19 samples)
4 - Proliferative DR: 46.15% (13 samples)
Model saved successfully: /content/IDRiD_EfficientNetB0_DR_Classifier.keras
Class names saved succes

In [42]:
# Bone Fracture

import kagglehub

# Download latest version
path = kagglehub.dataset_download("bmadushanirodrigo/fracture-multi-region-x-ray-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fracture-multi-region-x-ray-data' dataset.
Path to dataset files: /kaggle/input/fracture-multi-region-x-ray-data


In [43]:
os.listdir(path)

['README.dataset.txt', 'Bone_Fracture_Binary_Classification']

In [44]:
path = os.path.join(path,'Bone_Fracture_Binary_Classification')
# os.listdir(path)
path = os.path.join(path,'Bone_Fracture_Binary_Classification')
os.listdir(path)
val_path = os.path.join(path,'val')
test_path = os.path.join(path,'test')
train_path = os.path.join(path,'train')

os.listdir(train_path)

not_fractured_train_path = os.path.join(train_path,'not fractured')
fractured_train_path = os.path.join(train_path,'fractured')


not_fractured_test_path = os.path.join(test_path,'not fractured')
fractured_test_path = os.path.join(test_path,'fractured')


not_fractured_val_path = os.path.join(val_path,'not fractured')
fractured_val_path = os.path.join(val_path,'fractured')

In [46]:
os.listdir(fractured_train_path)

['9-rotated3-rotated1-rotated2 - Copy (1).jpg',
 '98-rotated1-rotated2-rotated1 (1).jpg',
 '79 - Copy (1).jpg',
 '1-rotated1-rotated1-rotated1 (1).jpg',
 '91-rotated1-rotated1-rotated2 (1).jpg',
 '98-rotated2-rotated1-rotated1 - Copy (1).jpg',
 '79-rotated3-rotated3-rotated3 - Copy.jpg',
 '81-rotated1-rotated3-rotated2.jpg',
 '87-rotated3-rotated2-rotated2 - Copy (1).jpg',
 '89-rotated2-rotated1 - Copy.jpg',
 '85-rotated3-rotated1-rotated3 - Copy.jpg',
 '79-rotated2-rotated2-rotated2 - Copy.jpg',
 '208.jpg',
 '96-rotated3-rotated2-rotated1 - Copy.jpg',
 '82-rotated2-rotated1-rotated2 (1).jpg',
 '1-rotated1-rotated2-rotated3 - Copy.jpg',
 '89-rotated3 - Copy (1).jpg',
 '18(1).jpg',
 '85-rotated3-rotated1-rotated1 - Copy (1).jpg',
 '86-rotated1-rotated1-rotated3.jpg',
 '81-rotated3-rotated3-rotated2 - Copy.jpg',
 '3-rotated2-rotated1-rotated2.jpg',
 '92-rotated3-rotated2-rotated3.jpg',
 '85-rotated2-rotated2-rotated1.jpg',
 '93-rotated2-rotated1-rotated2.jpg',
 '7-rotated1-rotated1 (1).j

In [50]:
import os, tensorflow as tf, kagglehub, numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

path=kagglehub.dataset_download("bmadushanirodrigo/fracture-multi-region-x-ray-data")
path=os.path.join(path,"Bone_Fracture_Binary_Classification","Bone_Fracture_Binary_Classification")
train_path,val_path,test_path=[os.path.join(path,x) for x in ["train","val","test"]]

def valid_files(folder):
    files=[]
    for root,_,names in os.walk(folder):
        for name in names:
            f=os.path.join(root,name)
            try:
                img=tf.io.read_file(f)
                tf.io.decode_image(img,channels=3,expand_animations=False)
                files.append(f)
            except:
                print("Skipping:",f)
    return files

train_files,val_files,test_files=valid_files(train_path),valid_files(val_path),valid_files(test_path)
print(len(train_files),len(val_files),len(test_files))

def make_ds(files,shuffle=False):
    paths=tf.constant(files)
    labels=tf.constant([0 if "not fractured" in f else 1 for f in files],dtype=tf.float32)
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    if shuffle: ds=ds.shuffle(len(files))
    def load(path,label):
        img=tf.io.decode_jpeg(tf.io.read_file(path),channels=3)
        img=tf.image.resize(img,(224,224))
        return img,label
    return ds.map(load,num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

train_ds=make_ds(train_files,True)
val_ds=make_ds(val_files)
test_ds=make_ds(test_files)

aug=tf.keras.Sequential([layers.RandomRotation(0.05),layers.RandomZoom(0.1),layers.RandomContrast(0.1)])

base=EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base.trainable=False

model=models.Sequential([layers.Input((224,224,3)),aug,base,layers.GlobalAveragePooling2D(),layers.BatchNormalization(),layers.Dense(128,activation="relu"),layers.Dropout(0.4),layers.Dense(1,activation="sigmoid")])

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=["accuracy",tf.keras.metrics.AUC(name="auc")])

callbacks=[ModelCheckpoint("/content/best_bone_model.keras",monitor="val_accuracy",save_best_only=True),EarlyStopping(monitor="val_accuracy",patience=5,restore_best_weights=True),ReduceLROnPlateau(monitor="val_loss",patience=2,factor=0.3)]

model.fit(train_ds,validation_data=val_ds,epochs=10,callbacks=callbacks)

base.trainable=True
for layer in base.layers[:-30]: layer.trainable=False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss="binary_crossentropy",metrics=["accuracy",tf.keras.metrics.AUC(name="auc")])
model.fit(train_ds,validation_data=val_ds,epochs=10,callbacks=callbacks)

model=tf.keras.models.load_model("/content/best_bone_model.keras")
loss,acc,auc=model.evaluate(test_ds)
print("Test Accuracy:",acc)
print("Test AUC:",auc)

y_true=np.concatenate([y.numpy().ravel() for _,y in test_ds])
y_pred=(model.predict(test_ds).ravel()>=0.5).astype(int)

print(classification_report(y_true,y_pred,target_names=["not fractured","fractured"]))
print(confusion_matrix(y_true,y_pred))
model.save("/content/Bone_Fracture_EfficientNetB0.keras")

Using Colab cache for faster access to the 'fracture-multi-region-x-ray-data' dataset.
Skipping: /kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/IMG0004347.jpg
Skipping: /kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/IMG0004148.jpg
Skipping: /kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/IMG0004134.jpg
Skipping: /kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/IMG0004149.jpg
Skipping: /kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification/train/not fractured/IMG0004143.jpg
Skipping: /kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/

16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 283ms/step
               precision    recall  f1-score   support

not fractured       0.98      1.00      0.99       262
    fractured       1.00      0.98      0.99       238

     accuracy                           0.99       500
    macro avg       0.99      0.99      0.99       500
 weighted avg       0.99      0.99      0.99       500

[[261   1]
 [  4 234]]


In [64]:
pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 7.4 MB/s eta 0:00:00


In [51]:
import requests

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": "diabetic retinopathy deep learning",
    "retmax": 100,
    "retmode": "json"
}

response = requests.get(url, params=params)
data = response.json()

pmids = data["esearchresult"]["idlist"]

print(len(pmids))
print(pmids[:10])

100
['42692194', '42688054', '42665469', '42659913', '42656234', '42651046', '42650958', '42640898', '42638907', '42631659']


In [52]:
# fetch the papers
import os
import requests
import xml.etree.ElementTree as ET

os.makedirs("data/raw", exist_ok=True)

search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": "diabetic retinopathy deep learning",
    "retmax": 100,
    "retmode": "json"
}

response = requests.get(search_url, params=params)
response.raise_for_status()

pmids = response.json()["esearchresult"]["idlist"]

fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "pubmed",
    "id": ",".join(pmids),
    "retmode": "xml"
}

response = requests.get(fetch_url, params=params)
response.raise_for_status()

root = ET.fromstring(response.text)

for i, article in enumerate(root.findall(".//PubmedArticle"), 1):
    pmid = article.findtext(".//PMID")
    title = article.findtext(".//ArticleTitle") or ""

    abstract_parts = article.findall(".//AbstractText")
    abstract = " ".join("".join(x.itertext()) for x in abstract_parts)

    text = f"""PMID: {pmid}

TITLE:
{title}

ABSTRACT:
{abstract}
"""

    with open(f"data/raw/paper_{i}.txt", "w", encoding="utf-8") as f:
        f.write(text)

print(f"Saved {len(pmids)} papers")

Saved 100 papers


In [56]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

raw_dir = "data/raw"

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

chunks = []

for filename in os.listdir(raw_dir):
  if not filename.endswith(".txt"):
    continue

  path = os.path.join(raw_dir,filename)

  with open(path,"r",encoding="utf-8") as f:
    text = f.read()

  document_chunks = splitter.split_text(text)

  for i,chunk in enumerate(document_chunks):
    chunks.append({"text":chunk,"source":filename,"chunk_id":i})

print(len(chunks))
print(chunks[0])


447
{'text': 'PMID: 42153778\n\nTITLE:\nScan Density Matters: Reproducibility of AI-Derived OCT Biomarkers in Diabetic Macular Edema.', 'source': 'paper_81.txt', 'chunk_id': 0}


In [58]:
from sentence_transformers import SentenceTransformer
import pickle

texts = [x["text"] for x in chunks]
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts,normalize_embeddings=True,show_progress_bar=True)

embeddings = np.array(embeddings)

os.makedirs("data/processed",exist_ok=True)

with open("data/processed/chunks.pkl","wb") as f:
  pickle.dump(chunks,f)

np.save("data/processed/embeddings.npy",embeddings)

print(len(chunks))
print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

447
(447, 384)


In [66]:
# put the vectors in faiss

import faiss
import numpy as np

embeddings = np.load("data/processed/embeddings.npy")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index,"data/processed/index.faiss")
print(index.ntotal)

447


In [67]:
import faiss
import pickle
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.read_index("data/processed/index.faiss")

with open("data/processed/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

query = "What deep learning methods are commonly used for diabetic retinopathy detection?"

query_vector = model.encode(
    [query],
    normalize_embeddings=True
)

scores, indices = index.search(query_vector, 5)

for score, index_id in zip(scores[0], indices[0]):
    print("\nSCORE:", score)
    print("SOURCE:", chunks[index_id]["source"])
    print(chunks[index_id]["text"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


SCORE: 0.8329215
SOURCE: paper_35.txt
PMID: 42479603

TITLE:
GENERALIZABILITY OF DEEP LEARNING MODELS FOR REFERABLE DIABETIC RETINOPATHY DETECTION: A CROSS-POPULATION STUDY.

SCORE: 0.8290924
SOURCE: paper_33.txt
PMID: 42491638

TITLE:
A systematic review on deep learning techniques for diabetic retinopathy classification in retinal fundus images.

SCORE: 0.8213005
SOURCE: paper_63.txt
Diabetic Retinopathy (DR) is a prominent results of diabetes mellitus that causes abnormalities lesions in retina. If not identified at early, it may progress to complete loss of vision. Unfortunately, DR is an irreversible, and treatment only sustains existing vision. Timely detection and accurate treatment of DR can considerably decrease the chance of blindness. Manual diagnosis of DR in retinal fundus images (RFIs) by ophthalmologist is time consuming, costly and laborious tasks with a higher risk of misdiagnosis. Recently, Deep learning (DL) has gained popularity and shown remarkable performance par

## Final: retrain and save lifestyle risk model + scaler

Runs the latest Kaggle dataset, evaluates a held-out split, then fits the deployment model on all rows. Keep the feature mappings aligned with `models/models.py`.


In [ ]:
from pathlib import Path
import kagglehub, joblib, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Download the user-selected latest Kaggle dataset.
path = kagglehub.dataset_download("miadul/lifestyle-and-health-risk-prediction")
dataset_path = Path(path)
df = pd.read_csv(dataset_path / "Lifestyle_and_Health_Risk_Prediction_Synthetic_Dataset.csv")
features = ["age", "weight", "height", "exercise", "sleep", "sugar_intake", "smoking", "alcohol", "married", "profession", "bmi"]
category_maps = {
    "exercise": {"none": 0, "low": 0, "medium": 1, "high": 2},
    "sugar_intake": {"low": 0, "medium": 1, "high": 2},
    "smoking": {"no": 0, "yes": 1}, "alcohol": {"no": 0, "yes": 1},
    "married": {"no": 0, "yes": 1},
    "profession": {"office_worker": 0, "teacher": 1, "artist": 2, "farmer": 3, "driver": 4, "engineer": 5, "student": 6, "doctor": 7, "other": 7},
}
X = df[features].copy()
for column, mapping in category_maps.items():
    X[column] = X[column].astype(str).str.lower().map(mapping).fillna(7 if column == "profession" else 0)
y = df["health_risk"].astype(str).str.lower().map({"low": 0, "high": 1})
if X.isna().any().any() or y.isna().any():
    raise ValueError("Dataset contains missing/unrecognized input values or labels.")

params = dict(n_estimators=300, learning_rate=0.03, max_depth=3, min_child_weight=3,
              gamma=0.1, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
              reg_lambda=2.0, objective="binary:logistic", eval_metric="logloss",
              random_state=42, n_jobs=-1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
eval_scaler = StandardScaler().fit(X_train)
eval_model = XGBClassifier(**params).fit(eval_scaler.transform(X_train), y_train)
y_pred = eval_model.predict(eval_scaler.transform(X_test))
print("Held-out accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=["Low", "High"]))

# Refit both artifacts together on all available rows for deployment.
scaler = StandardScaler().fit(X)
model = XGBClassifier(**params).fit(scaler.transform(X), y)
output_dir = Path("weights") if Path("weights").is_dir() else Path(".")
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, output_dir / "xgboost_risk.joblib")
joblib.dump(scaler, output_dir / "xgboost_risk_scaler.joblib")
print("Saved paired model and scaler to", output_dir.resolve())